In [1]:
# Install plotly
!pip install plotly --quiet

import zipfile
import os

# Extract the zip file
with zipfile.ZipFile('archive (4).zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("✅ Done! Files:")
for f in os.listdir('.'):
    print(f)

✅ Done! Files:
.config
license.txt
googleplaystore.csv
archive (4).zip
googleplaystore_user_reviews.csv
sample_data


In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv('googleplaystore.csv')

# Drop duplicates and nulls
df.drop_duplicates(inplace=True)
df.dropna(subset=['Category', 'Rating', 'Reviews', 'Installs'], inplace=True)

# Clean Installs → numeric
df['Installs'] = df['Installs'].str.replace('[+,]', '', regex=True)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
df.dropna(subset=['Installs'], inplace=True)
df['Installs'] = df['Installs'].astype(int)

print("✅ Data loaded! Shape:", df.shape)
df.head()

✅ Data loaded! Shape: (8892, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,10000,Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,5000000,Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,50000000,Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,100000,Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [3]:
# Group by Category and sum installs
category_df = df.groupby('Category').agg(
    Total_Installs=('Installs', 'sum')
).reset_index()

# Filter 1: Installs > 1 million
category_df = category_df[category_df['Total_Installs'] > 1_000_000]

# Filter 2: Category should NOT start with A, C, G, or S
category_df = category_df[~category_df['Category'].str.startswith(('A', 'C', 'G', 'S'))]

# Filter 3: Top 5 categories by installs
top5 = category_df.nlargest(5, 'Total_Installs')

print("✅ Top 5 Categories:")
print(top5)

✅ Top 5 Categories:
            Category  Total_Installs
25      PRODUCTIVITY     12463070180
29             TOOLS     11450724500
11            FAMILY     10041130590
24       PHOTOGRAPHY      9721243130
30  TRAVEL_AND_LOCAL      6361859300


In [7]:
import plotly.express as px
from datetime import datetime
import pytz

ist = pytz.timezone('Asia/Kolkata')
current_hour = datetime.now(ist).hour

print(f"Current IST hour: {current_hour}")

if not (18 <= current_hour < 20):
    print("⛔ Chart is only available between 6 PM and 8 PM IST. Please come back then.")

else:
    # Map top5 categories to countries using ISO-3 codes
    country_codes = ['USA', 'IND', 'GBR', 'DEU', 'BRA']

    top5_reset = top5.reset_index(drop=True)
    top5_reset['Country_Code'] = country_codes

    # Highlight categories where installs > 1 million
    top5_reset['Highlight'] = top5_reset['Total_Installs'].apply(
        lambda x: 'Exceeds 1M' if x > 1_000_000 else 'Below 1M'
    )

    fig = px.choropleth(
        top5_reset,
        locations='Country_Code',
        color='Total_Installs',
        hover_name='Category',
        hover_data={'Total_Installs': True, 'Highlight': True},
        color_continuous_scale='Viridis',
        title='Global Installs by Top 5 App Categories<br>(Installs > 1M | Excluding A, C, G, S)',
        labels={'Total_Installs': 'Total Installs'}
    )

    fig.update_layout(
        title_font_size=16,
        geo=dict(showframe=False, showcoastlines=True)
    )

    fig.show()
    fig.write_html('task2_choropleth.html')
    print("✅ Chart saved!")

Current IST hour: 18


✅ Chart saved!
